# Lecture: How Convolutional Neural Networks See (CNNs on CIFAR-10)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain what a convolution, a filter, and a feature map are in one sentence each (§4).
- Explain why one shared filter beats a flat MLP, and predict output sizes with stride and padding (§5).
- Read the system map (§8) and narrate one forward pass: image → conv blocks → head, plus the 4×4 → pool trace.


## The big idea

Imagine reading a big poster through a tiny 3×3 cardboard window that you slide across it, asking at every spot: "is there a vertical edge here?" A **convolution** is exactly that sliding window: one small grid of numbers (the **filter**, also called the **kernel**) is stamped over every patch of the image, and each stamp produces one number in the output picture (the **feature map**, bright where the filter's pattern was found). Because the *same* filter is reused at every position (**weight sharing**), the network needs only 9 numbers to find vertical edges anywhere in the photo, instead of learning a separate detector per pixel the way a flat MLP would. After each convolution the lab shrinks the map (**max-pooling**: keep only the brightest pixel of each 2×2 window), bends the signal (**ReLU**: negatives become zero) and rescales it (**BatchNorm**: re-center each channel so training stays stable). Stack two such blocks and each deep activation ends up summarizing a 10×10 patch of the original 32×32 photo — that patch is its **receptive field** — and a final linear layer votes those local clues into 10 class scores.


In [ ]:
import numpy as np  # bring in numpy for the tiny image and filter math
import matplotlib.pyplot as plt  # bring in pyplot for the three picture panels
np.random.seed(7)  # fix the seed so the demo is identical every run
img = np.zeros((12, 12))  # make a tiny 12x12 black canvas as our toy photo
img[:, 5:7] = 1.0  # paint a bright vertical bar in columns 5 and 6
img[2, 2] = 0.8  # add one dimmer noisy pixel for pooling to squash later
filt = np.array([[-1.0, 0.0, 1.0],  # filter top row: subtract left, ignore middle, add right
                 [-1.0, 0.0, 1.0],  # filter middle row: the same left-minus-right recipe
                 [-1.0, 0.0, 1.0]])  # filter bottom row: same recipe, a vertical-edge finder
out = np.zeros((10, 10))  # allocate the feature map: 12 - 3 + 1 = 10 stamps per side
for i in range(10):  # slide the window over the 10 output rows
    for j in range(10):  # slide the window over the 10 output columns
        patch = img[i:i + 3, j:j + 3]  # slice the 3x3 patch currently under the window
        out[i, j] = float((patch * filt).sum())  # stamp: multiply-accumulate filter against patch
print('max response:', round(float(out.max()), 2))  # strongest edge signal
print('flat patch response:', round(float(out[9, 0]), 2))  # bottom-left patch (rows 9-11, all black) stamps to 0.0; the bright bar edge elsewhere gives 3.0 (see max above)
print('corner patch response:', round(float(out[0, 0]), 2))  # corner patch overlaps the 0.8 noise pixel at img[2,2] (filter corner weight -1.0 x 0.8): prints 0.8, not 0.0
fig, ax = plt.subplots(1, 3, figsize=(9, 3))  # build three side-by-side picture panels
ax[0].imshow(img, cmap='gray', vmin=0, vmax=1)  # left panel: the toy photo with its bright bar
ax[0].set_title('input: bright vertical bar')  # title the input panel
ax[1].imshow(filt, cmap='coolwarm', vmin=-1, vmax=1)  # middle panel: the edge-finder filter itself
ax[1].set_title('filter: vertical-edge finder')  # title the filter panel
im = ax[2].imshow(out, cmap='coolwarm')  # right panel: the feature map of edge responses
ax[2].set_title('output: edges light up')  # title the feature-map panel
fig.colorbar(im, ax=ax[2], shrink=0.8)  # add a colorbar so response values are readable
fig.suptitle('Convolution = slide the filter, stamp one number per patch')  # one-line moral of the story
fig.tight_layout()  # snug the layout so titles do not overlap
plt.show()  # render the three panels


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 stamps the filter, §5 counts the savings, §6 shrinks and bends, §7 grows the window.

| Term | One-liner | Section |
|---|---|---|
| Convolution | Sliding a small filter over the image, one number stamped per patch | §4 |
| Filter (kernel) | The small grid of learned numbers doing the stamping (e.g. 3×3 = 9 numbers) | §4 |
| Feature map | The output picture of stamps: bright where the pattern was found | §4 |
| Weight sharing | Reusing the *same* filter at every position, so edges are found anywhere cheaply | §5 |
| Stride | How many pixels the window jumps per stamp (stride 2 = smaller output) | §5 |
| Padding | Extra border of zeros so the output keeps the input's size | §5 |
| Max-pooling | Shrinking a map by keeping only the brightest pixel in each 2×2 window | §6 |
| ReLU | The bend: negatives become zero, positives pass through | §6 |
| BatchNorm | Per-channel re-centering per batch so training stays stable and fast | §6 |
| Receptive field | How many input pixels one deep activation can "see" (lab: 10×10 of 32×32) | §7 |


### §4 Convolution, filter, feature map

**In one sentence: a convolution slides one small filter over the image and stamps a single number per patch; the filter is the small grid of learned numbers, and the feature map is the output picture of those stamps, bright exactly where the filter's pattern matched.**

The stamp is just a dot product: lay the filter over a same-sized patch, multiply overlapping entries, add everything up. A vertical-edge filter like `[-1, 0, +1]` columns gives ~0 on flat regions (left equals right, they cancel) and ±3.0 on a sharp black-to-white edge (one side bright, the other dark). Watch it: a hand-worked 2×2 filter on a 4×4 photo — the top-left patch `[[3, 1], [0, 2]]` stamps `3×1 + 1×0 + 0×1 + 2×0 = 3`, and all nine stamps form the 3×3 feature map `[[3, 3, 1], [1, 2, 5], [3, 1, 4]]`.


In [ ]:
import numpy as np  # reuse numpy for the hand-worked stamping math
import matplotlib.pyplot as plt  # reuse pyplot for the before/after pictures
np.random.seed(7)  # fix the seed for reproducibility
x = np.array([[3., 1., 0., 2.],  # row 0 of our tiny 4x4 photo
              [0., 2., 1., 0.],  # row 1 of the photo
              [1., 0., 4., 1.],  # row 2 of the photo
              [2., 1., 0., 3.]])  # row 3 of the photo
w = np.array([[1., 0.],  # filter top row: keep the left pixel, ignore the right one
              [1., 0.]])  # filter bottom row: keep the left pixel too (a toy left-column adder)
y = np.zeros((3, 3))  # allocate the 3x3 feature map: 4 - 2 + 1 stamps per side
for i in range(3):  # walk the 3 output rows
    for j in range(3):  # walk the 3 output columns
        y[i, j] = float((x[i:i + 2, j:j + 2] * w).sum())  # stamp: dot the filter against this 2x2 patch
print('top-left patch stamps:', round(float(y[0, 0]), 1), '(= 3x1 + 1x0 + 0x1 + 2x0)')  # worked patch: left column 3 + 0
print('full feature map y =')  # label the printed 3x3 output
print(y)  # all nine stamps: rows [3,3,1] / [1,2,5] / [3,1,4]
print('shapes: 4x4 photo -> 3x3 map (one stamp per 2x2 patch)')  # sizes shrink because each stamp eats a patch
fig, ax = plt.subplots(1, 2, figsize=(7, 3))  # make two panels: photo vs feature map
ax[0].imshow(x, cmap='gray')  # draw the 4x4 photo as a picture
ax[0].set_title('input photo (4x4)')  # title the left panel
ax[1].imshow(y, cmap='coolwarm')  # draw the 3x3 feature map as a picture
ax[1].set_title('feature map (3x3): left-column sums')  # title the right panel
fig.suptitle('One filter, nine stamps: convolution by hand')  # overall takeaway
fig.tight_layout()  # tidy spacing between the panels
plt.show()  # render the figure


### §5 Weight sharing, stride, padding

**In one sentence: weight sharing reuses the *same* filter at every position (so one 3×3 filter costs 9 weights no matter how big the photo is), stride sets how many pixels the window jumps per stamp, and padding adds a zero border so the map keeps the input's size.**

The savings are extreme: the lab's first layer (3 input channels → 32 filters of 3×3) learns just 896 numbers, while a flat MLP doing the same 32×32×3 → 32×32×32 shape change would need 100,696,064 — about 112,384× more. Output sizes follow one formula: `out = (W − K + 2P) / S + 1` (W = input width, K = filter width, P = padding, S = stride). Watch it: the bars show 896 vs ~100.7M on a log scale, and the prints check the formula — stride 1 with padding 1 keeps 32→32, stride 2 halves 32→16, no padding shrinks 32→30.


In [ ]:
import numpy as np  # reuse numpy for the parameter-count arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the shared-vs-dense bars
np.random.seed(7)  # fix the seed for reproducibility
conv_params = 32 * (3 * 3 * 3) + 32  # shared conv 3->32, k3: 32 filters x (3 channels x 9) + 32 biases
dense_in = 32 * 32 * 3  # flat MLP input size: every pixel x every channel = 3072
dense_out = 32 * 32 * 32  # flat MLP output size: same spatial shape with 32 channels = 32768
dense_params = dense_in * dense_out + dense_out  # dense weights plus one bias per output
print('shared conv params:', conv_params)  # expect 896: tiny because one filter serves all positions
print('flat dense params:', dense_params)  # expect 100696064: a private weight per pixel pair
print('sharing saves ~x', round(dense_params / conv_params))  # expect ~112384: the point of §5
def out_size(W, K, P, S):  # the one size formula: (width - kernel + 2*pad) // stride + 1
    return (W - K + 2 * P) // S + 1  # integer math: sizes are whole stamps, no fractions
print('32,k3,pad1,stride1 ->', out_size(32, 3, 1, 1))  # expect 32: padding preserves the size
print('32,k3,pad1,stride2 ->', out_size(32, 3, 1, 2))  # expect 16: stride 2 halves each side
print('32,k3,pad0,stride1 ->', out_size(32, 3, 0, 1))  # expect 30: no padding loses a border ring
fig, ax = plt.subplots(figsize=(6, 4))  # make one panel for the savings bars
ax.bar(['shared conv', 'flat dense'], [conv_params, dense_params], color=['steelblue', 'tomato'])  # two bars, same job
ax.set_yscale('log')  # log scale: the 100,000x gap would hide the blue bar otherwise
ax.set_ylabel('learned parameters (log scale)')  # label: fewer params = cheaper + less overfit
ax.set_title('Weight sharing: 896 vs ~100.7M for the same shape change')  # title the takeaway
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §6 Max-pooling, ReLU, BatchNorm

**In one sentence: max-pooling shrinks each map by keeping only the brightest pixel of every 2×2 window, ReLU bends the signal with `max(0, z)` so the network can learn curves, and BatchNorm re-centers each channel per batch so training stays stable.**

Pooling does three jobs at once: it keeps the strongest detection (the 0.8 noise pixel vanishes if a brighter neighbour shares its window), it halves the compute downstream, and it tolerates small shifts (move the edge one pixel and the same max usually wins). ReLU then zeroes negatives — on `[-1.0, 0.0, 0.5, 2.0]` it gives `[0, 0, 0.5, 2.0]` with a backward mask of `[0, 0, 1, 1]` (dead entries pass no gradient). BatchNorm finishes the trio: for a channel reading `[2, 4, 6]` across the batch (mean 4.0), it outputs `[-1.225, 0, 1.225]` — same pattern, unit scale, so the next layer always receives well-behaved inputs. Watch it: our 4×4 photo pools to `[[3, 2], [2, 4]]`, half the size with the peaks intact.


In [ ]:
import numpy as np  # reuse numpy for pooling, ReLU, and BatchNorm math
import matplotlib.pyplot as plt  # reuse pyplot for the before/after pooling pictures
np.random.seed(7)  # fix the seed for reproducibility
x = np.array([[3., 1., 0., 2.],  # row 0: same running 4x4 photo as §4
              [0., 2., 1., 0.],  # row 1 of the photo
              [1., 0., 4., 1.],  # row 2 of the photo
              [2., 1., 0., 3.]])  # row 3 of the photo
pool = np.zeros((2, 2))  # allocate the 2x2 shrunk map (each entry covers a 2x2 window)
for i in range(2):  # walk the 2 pooled rows
    for j in range(2):  # walk the 2 pooled columns
        pool[i, j] = x[2 * i:2 * i + 2, 2 * j:2 * j + 2].max()  # keep the brightest pixel of each window
print('max-pooled 4x4 -> 2x2 =')  # label the shrunk map
print(pool)  # expect [[3, 2], [2, 4]]: peaks survive, size halves
vals = np.array([-1.0, 0.0, 0.5, 2.0])  # four toy activations entering ReLU
print('ReLU in :', vals)  # show the inputs
print('ReLU out:', np.maximum(0, vals))  # expect [0, 0, 0.5, 2]: negatives flattened
print('ReLU mask:', (vals > 0).astype(int))  # expect [0, 0, 1, 1]: backward passes only live entries
ch = np.array([2.0, 4.0, 6.0])  # one channel's values across a 3-sample batch
mu = ch.mean()  # batch mean: 4.0, the center to subtract
sd = ch.std()  # batch std: ~1.633, the scale to divide by
print('BatchNorm of [2,4,6]:', np.round((ch - mu) / sd, 3))  # expect [-1.225, 0, 1.225]: same shape, unit scale
fig, ax = plt.subplots(1, 2, figsize=(7, 3))  # make two panels: before vs after pooling
ax[0].imshow(x, cmap='gray')  # draw the 4x4 input as a picture
ax[0].set_title('before pooling (4x4)')  # title the left panel
ax[1].imshow(pool, cmap='gray')  # draw the 2x2 pooled map as a picture
ax[1].set_title('after max-pool (2x2): peaks survive')  # title the right panel
fig.suptitle('Pooling keeps the strongest signal, halves the picture')  # overall takeaway
fig.tight_layout()  # tidy spacing of the two panels
plt.show()  # render the figure


### §7 Receptive field — how much each neuron sees

**In one sentence: the receptive field is the patch of the original photo that one deep activation can "see" — it starts at 1 pixel and grows with every convolution and pooling layer.**

The growth rule is simple: a layer with kernel `k` adds `(k − 1) × (current stride product)` new pixels, and pooling doubles the stride product for everything after it. The lab stacks conv(3, stride 1) → pool(2, stride 2) → conv(3, stride 1) → pool(2, stride 2): the field walks 1 → 3 → 4 → 8 → 10, so each classifier input summarizes a 10×10 patch of the 32×32 photo — local evidence, not the whole scene. That locality is a feature (edges and textures are local) and the reason the final linear layer exists: it combines many local opinions into one global vote. Watch it: the step plot draws the 1 → 3 → 4 → 8 → 10 staircase.


In [ ]:
import numpy as np  # reuse numpy for the receptive-field walk
import matplotlib.pyplot as plt  # reuse pyplot for the growth staircase
np.random.seed(7)  # fix the seed for reproducibility
rf, stride = 1, 1  # start: each activation sees exactly 1 input pixel, stride product 1
rfs, labels = [rf], ['input']  # record the field after each stage for the plot
for name, k, s in [('conv3', 3, 1), ('pool2', 2, 2), ('conv3', 3, 1), ('pool2', 2, 2)]:  # lab order: conv,pool,conv,pool
    rf = rf + (k - 1) * stride  # grow the field by (k-1) input-pixels at current stride
    stride = stride * s  # fold this layer's stride into the running product
    rfs.append(rf)  # record the new field size
    labels.append(name)  # record the stage name
    print(name, '-> field', rf, 'x', rf, ', stride product', stride)  # narrate each growth step
print('final: 10x10 of 32x32 (local evidence, not the whole photo)')  # the headline: 10, not 32
fig, ax = plt.subplots(figsize=(7, 4))  # make one panel for the staircase
ax.step(range(len(rfs)), rfs, where='mid', linewidth=2)  # draw the 1 -> 3 -> 4 -> 8 -> 10 staircase
ax.plot(range(len(rfs)), rfs, 'o')  # dot each stage so values are readable
ax.set_xticks(range(len(labels)))  # one tick per stage
ax.set_xticklabels(labels)  # label ticks with stage names
ax.set_ylabel('receptive field (pixels per side)')  # label: patch width one activation sees
ax.set_title('Deeper = wider view: 1 -> 3 -> 4 -> 8 -> 10')  # title the growth story
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## How it all fits together — the system map (§8)

One forward pass is an assembly line where every § term holds one station. The 32×32×3 photo enters **conv block 1** (§4 stamps with 32 shared filters per §5, then §6's BatchNorm → ReLU → pool), leaving a 16×16×32 stack of edge maps. **Conv block 2** repeats the recipe with 64 filters and pools to 8×8×64 — each entry now seeing a 10×10 input patch (§7). The **head** flattens those 8×8×64 = 4096 clues into 10 class scores. The trace below replays the whole line on our toy 4×4 photo: 4×4 → 3×3 stamps (one border ring eaten) → ReLU (no-op here, all non-negative) → while pooling halves a 4×4 map to 2×2 peaks `[[3, 2], [2, 4]]`.


In [ ]:
import numpy as np  # import numpy for the seed habit (diagram uses no randomness)
import matplotlib.pyplot as plt  # only pyplot draws: this cell is a pure diagram
np.random.seed(7)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('photo\n32x32x3', 1), ('conv1+BN+ReLU\n§4-§6', 3.6), ('pool\n16x16x32', 6.2), ('conv2+BN+ReLU\npool 8x8x64', 8.8), ('flatten+Linear\n10 logits', 11.6)]  # pipeline stations
colors = ['wheat', 'lightblue', 'lightgreen', 'lightblue', 'plum']  # color per station type
for (name, xc), c in zip(stages, colors):  # draw each pipeline station
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor=c))  # rounded colored box around it
ax.text(7, 3.5, 'FORWARD: space shrinks (32 -> 16 -> 8), channels grow (3 -> 32 -> 64)', ha='center', fontsize=9, fontweight='bold')  # banner line
for xa, xb in [(1.9, 2.7), (4.6, 5.4), (7.0, 7.9), (9.9, 10.7)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2), arrowprops=dict(arrowstyle='->'))  # forward arrow left -> right
ax.text(7, 0.9, 'field grows 1 -> 3 -> 4 -> 8 -> 10 (§7); head votes local clues into classes', ha='center', fontsize=9, color='darkred')  # field note
ax.set_title('System map: one forward pass through every vocabulary term')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the end-to-end numeric trace
np.random.seed(7)  # fix the seed (arrays below are hardcoded; habit kept)
x = np.array([[3., 1., 0., 2.], [0., 2., 1., 0.], [1., 0., 4., 1.], [2., 1., 0., 3.]])  # toy 4x4 photo (§4)
w = np.array([[1., 0.], [1., 0.]])  # toy 2x2 shared filter (§4, §5: same weights everywhere)
z = np.zeros((3, 3))  # allocate the convolution output
for i in range(3):  # stamp rows (§4)
    for j in range(3):  # stamp columns (§4)
        z[i, j] = float((x[i:i + 2, j:j + 2] * w).sum())  # one shared-filter stamp
a = np.maximum(0, z)  # ReLU (§6): all stamps are non-negative, so nothing changes here
pool = np.zeros((2, 2))  # allocate the pooled map (§6: pooling halves 4x4 -> 2x2)
for i in range(2):  # pooled rows
    for j in range(2):  # pooled columns
        pool[i, j] = x[2 * i:2 * i + 2, 2 * j:2 * j + 2].max()  # keep each photo window's peak
print('conv 4x4 -> 3x3 =')  # label the stamp stage
print(z)  # expect rows [3,3,1] / [1,2,5] / [3,1,4]
print('after ReLU (unchanged, all >= 0) =')  # label the bend stage
print(a)  # identical to z: the bend only bites on negatives
print('meanwhile pooling halves 4x4 -> 2x2 =')  # label the shrink stage (stamps trim, pooling halves)
print(pool)  # expect [[3, 2], [2, 4]]: the trace's final shrunk answer
print('lab scale: 32 ->(conv,pad1) 32 ->(pool) 16 ->(conv,pad1) 16 ->(pool) 8')  # same dance on real photos


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, on real 32×32 photos:

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `np.random.seed` habit as this lecture.
2. **Cell 2 (data)** — loads a 4000-image CIFAR-10 subset (32×32 photos, 10 classes) or a synthetic fallback — the "posters" our sliding windows will read.
3. **Cell 3 (from-scratch convolution)** — the lab's `conv2d_naive` is the exact sliding-window loop of §4, checked against torch (max diff ~1e-6), then `TinyCNN` stacks two Conv → BatchNorm → ReLU → Pool blocks (§4–§6) plus a linear voting head.
4. **Cell 4 (training loop)** — 5 epochs with Adam and cross-entropy: watch train accuracy climb from chance (0.10) toward ~0.55 as the filters learn edge/shape detectors.
5. **Cell 5 (evaluation)** — test accuracy (~0.44) with the loss/accuracy curves in `results/curves.png`; the gap to train accuracy is the generalization question.
6. **Cell 6 (receptive field + ablations)** — the hand-computed 10×10 field of §7 (each classifier input sees a 10×10 patch, not the whole photo), a k3-vs-k5 params/speed check (§5's tradeoff), and a no-BatchNorm rerun showing what §6's rescaling buys.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Why does a CNN need ~112,384× fewer weights than a flat MLP for the same shape change?**

*Answer: weight sharing (§5) — one 3×3 filter (9 weights, 896 for the whole 3→32 layer) is stamped everywhere, while the MLP needs a private weight per pixel pair (100,696,064). Recompute both bars in the §5 cell.*

2. **What does max-pooling do to our 4×4 photo, and why is that useful?**

*Answer: it keeps each 2×2 window's peak, giving `[[3, 2], [2, 4]]` (§6) — half the size with the strongest detections intact, which also makes later layers' strides (and hence §7's field) grow faster.*

3. **Why is the lab's receptive field only 10×10 of a 32×32 image, and who finishes the job?**

*Answer: each 3×3 conv adds a little and each pool doubles the step, walking 1 → 3 → 4 → 8 → 10 (§7) — so classifier inputs hold local evidence, and the final linear head (§8) combines those local opinions into one global vote.*
